# Lesson 5.3 — Compliance and Ethical AI

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Map regulatory requirements (GDPR, HIPAA, FedRAMP) to architectural controls
- Implement bias/fairness evaluation across demographic slices
- Document transparency and data-handling obligations

**You will produce:** A compliance control matrix and a fairness-by-slice evaluation.

## Concepts

Architects translate regulations into **controls**:

- **GDPR** (EU personal data) — lawful basis, data minimization, right to erasure, data-residency, and limits on automated decisions. Implies PII redaction, retention limits, and audit trails.
- **HIPAA** (US health data / PHI) — safeguards for protected health information, access controls, audit logs, and a Business Associate Agreement with vendors.
- **FedRAMP** (US government cloud) — authorized cloud environments and strict control baselines.

**Ethical AI** adds bias/fairness, transparency, and accountability. **Bias** means systematically worse outcomes for some groups; you test it by **slicing** evaluation metrics across demographics and checking for gaps. **Transparency** means documenting data use, model limits, and how decisions are made/overridden. These obligations shape data flow, logging, storage region, and HITL — verify specifics with counsel and the official docs; this lesson builds the scaffolding.

> **Reference:** [Anthropic — Trust, safety & responsible use](https://docs.anthropic.com/en/docs/overview)

In [ ]:
# !pip install anthropic
import os, json

try:
    import anthropic
    if os.environ.get("ANTHROPIC_API_KEY"):
        client = anthropic.Anthropic(); LIVE = True
    else:
        client = None; LIVE = False
except Exception as e:
    client = None; LIVE = False
    print("anthropic SDK unavailable:", e)

MODELS = {"opus":"claude-opus-4-20250514","sonnet":"claude-sonnet-4-20250514","haiku":"claude-3-5-haiku-20241022"}

def ask_claude(prompt, model=MODELS["sonnet"], system=None, max_tokens=512):
    if not LIVE:
        return f"[MOCK {model}] would answer: {prompt[:80]}..."
    msg = client.messages.create(model=model, max_tokens=max_tokens,
        system=system or "You are a helpful assistant.",
        messages=[{"role":"user","content":prompt}])
    return msg.content[0].text

print("LIVE" if LIVE else "MOCK", "mode ready")

## Code-Along: Step 1 — Build a compliance control matrix

We map each regulation to the concrete architectural controls it demands, producing an auditable matrix.

In [ ]:
compliance = {
  "GDPR":   ["lawful basis recorded","PII minimization","right-to-erasure workflow","EU data residency","human review of automated decisions"],
  "HIPAA":  ["PHI access controls","audit logging","encryption at rest/in transit","BAA with vendors","minimum necessary use"],
  "FedRAMP":["authorized cloud boundary","control baseline","continuous monitoring","incident response"],
}
with open("compliance_matrix.json","w",encoding="utf-8") as fh:
    json.dump(compliance, fh, indent=2)
for reg, controls in compliance.items():
    print(reg); [print('  -',c) for c in controls]

## Code-Along: Step 2 — Evaluate fairness across slices

We compute accuracy per demographic slice and flag gaps above a tolerance. A single aggregate accuracy can hide serious disparities — slicing surfaces them.

In [ ]:
# (slice_label, correct, total) from your eval harness, grouped by demographic
slices = [("group_A", 92, 100), ("group_B", 78, 100), ("group_C", 90, 100)]
def fairness_report(slices, tolerance=0.05):
    accs = {g: c/t for g,c,t in slices}
    gap = max(accs.values()) - min(accs.values())
    flagged = gap > tolerance
    return accs, gap, flagged

accs, gap, flagged = fairness_report(slices)
for g,a in accs.items(): print(f"{g}: {a:.2f}")
print(f"max gap={gap:.2f} -> {'FAIRNESS CONCERN' if flagged else 'within tolerance'}")

## Exercise

Extend compliance handling: write a `data_flow_check(record)` that, given a data record's fields and destination, flags violations (e.g. PHI sent to a non-BAA service, EU data stored outside the EU). Then propose one mitigation per flagged violation. Add a comment that regulatory specifics must be confirmed with legal/compliance, not inferred.

In [ ]:
# TODO: data_flow_check(record) -> list of violations
# record example: {'contains_phi': True, 'destination': 'vendor_X', 'has_baa': False, 'region':'US','is_eu_subject':True}
# TODO: propose a mitigation per violation

## Key Takeaways

- Translate GDPR/HIPAA/FedRAMP into concrete, auditable architectural controls.
- Compliance shapes data flow, logging, storage region, retention, and HITL.
- Test fairness by slicing metrics across groups — aggregates hide disparities.
- Document transparency and confirm regulatory specifics with counsel, not guesses.

---

**Next:** [Lesson 5.4 — Discovery and Communicating Trade-offs](./lesson-5.4-discovery-communicating-tradeoffs.ipynb)